# 少榆0616ver_5：Past / Current / Future 整合報告

## 報告主軸

這版是在 `少榆0616ver_4` 的 DB 回寫主線上，補上 past / current 整合：

```text
Database/versionB sensor_1min / sensor_3min
-> IntegratedSprayLineService 取得 past window / current snapshot
-> 輸出 UI 需要的 time-series 格式
-> current / future 接少榆 MonitoringWorker + FutureService
-> alert_event / batch_station_status / future_prediction_result 仍回寫 Database/versionB
```

重點：  
**past / current 的資料流有整合進來，但 DB persistence 沒有改成 runtime JSON，也沒有改成 HTTP endpoint。**


In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "webservices").exists():
    candidates = list(Path.cwd().glob("**/少榆0616ver_5"))
    if candidates:
        PROJECT_DIR = candidates[0]

print("PROJECT_DIR =", PROJECT_DIR)
print("webservices exists:", (PROJECT_DIR / "webservices").exists())

def show_file(path, start=1, end=None, keyword=None):
    p = PROJECT_DIR / path
    print("=" * 90)
    print(path)
    print("=" * 90)
    if not p.exists():
        print("找不到檔案：", p)
        return

    lines = p.read_text(encoding="utf-8", errors="replace").splitlines()

    if keyword:
        hits = [i for i, line in enumerate(lines, 1) if keyword in line]
        if not hits:
            print("找不到 keyword:", keyword)
            return
        start = max(1, hits[0] - 6)
        end = min(len(lines), hits[0] + 22)

    if end is None:
        end = min(len(lines), start + 70)

    for i, line in enumerate(lines[start-1:end], start):
        print(f"{i:04d}: {line}")


## 1. 新增整合服務入口

### 要開檔案
`webservices/integrated_service/sprayline_integrated_service.py`

### 主旨
把 past / current / future 對齊成同一個 Python service 流程。

### 報告講法
老師，`0616ver_4` 已經完成 Future / Alert / DB 回寫；這版補上 past/current 的資料入口。  
我沒有把同學的 WebServices 整包搬進來，而是把 time slider 概念和 UI time-series 格式整合到正式 DB 流程上。


In [ ]:
show_file("webservices/integrated_service/sprayline_integrated_service.py", start=1, end=95)


## 2. Time slider 概念

### 要開檔案
`webservices/integrated_service/sprayline_integrated_service.py`

### 規則
```text
slider_value < 0  -> past
slider_value == 0 -> current
slider_value > 0  -> future
```

### 報告講法
這裡對齊 past/current 同學的 time slider 概念。  
差別是我這邊的資料來源仍走 Database/versionB，後段仍接少榆的 alert / future DB 回寫。


In [ ]:
show_file("webservices/integrated_service/sprayline_integrated_service.py", keyword="def build_time_window")


## 3. Past window 資料取得

### 要開檔案
`webservices/integrated_service/sprayline_integrated_service.py`

### 主旨
past 模式會以 slider 指定的時間點作為 anchor，往前抓一段 window。

### 報告講法
past 不是用假資料，也不是 runtime JSON，而是透過 `Database/versionB` 的 `query_sensor_1min` / `query_sensor_3min` 查指定時間窗。


In [ ]:
show_file("webservices/integrated_service/sprayline_integrated_service.py", keyword="def fetch_sensor_window")


## 4. Current snapshot

### 要開檔案
`webservices/integrated_service/sprayline_integrated_service.py`

### 主旨
current snapshot 由最近 window 內最新的 `sensor_1min` 和 `sensor_3min` 合併。

### 報告講法
current snapshot 是 UI 畫面和後續 alert / future model 的目前狀態輸入。


In [ ]:
show_file("webservices/integrated_service/sprayline_integrated_service.py", keyword="def build_current_snapshot")


## 5. UI 需要的 time-series 格式

### 要開檔案
`webservices/integrated_service/sprayline_integrated_service.py`

### 主旨
輸出 UI 可用格式：viewer_state、stations、time_series points、past_window summary、current_snapshot、future_prediction_payload。

### 報告講法
這裡是給 UI 組後續接資料用，不是讓前端直接連 PostgreSQL。  
UI 可以拿 `time_series.points` 畫趨勢，拿 `current_snapshot` 顯示目前狀態。


In [ ]:
show_file("webservices/integrated_service/sprayline_integrated_service.py", keyword="def build_ui_time_series_response")


## 6. Future 與 DB 回寫仍保持少榆 / Database/versionB 主線

### 要開檔案
`webservices/integrated_service/sprayline_integrated_service.py`

### 主旨
future prediction payload 仍對齊 `future_prediction_result`，寫回仍用 `db_future.insert_future_prediction_result`。

### 報告講法
past/current 是前段資料入口；future/alert/status 的 DB persistence 沒有被 time_series demo JSON 取代。


In [ ]:
show_file("webservices/integrated_service/sprayline_integrated_service.py", keyword="def estimate_future_from_snapshot")


## 7. 可選 DB write-back

### 要開檔案
`webservices/integrated_service/sprayline_integrated_service.py`

### 主旨
`write_back=False` 時只輸出 UI time-series；`write_back=True` 時才觸發 MonitoringWorker / FutureService 寫 DB。

### 報告講法
這樣可以區分「UI 查詢」和「正式寫回 DB」，避免每次 UI 拖 slider 都造成 DB 寫入。


In [ ]:
show_file("webservices/integrated_service/sprayline_integrated_service.py", keyword="def run_integrated_once")


## 8. 實測腳本

### 要開檔案
`scripts/run_past_current_integrated_demo.py`

### 用法
```powershell
python scripts\run_past_current_integrated_demo.py --slider 0 --station Station_1
python scripts\run_past_current_integrated_demo.py --slider -60 --window 30 --station Station_1
python scripts\run_past_current_integrated_demo.py --slider 30 --station Station_1
python scripts\run_past_current_integrated_demo.py --slider 30 --station Station_1 --write-back
```

### 報告講法
這支腳本可以測 current、past、future 三種 slider 狀態。  
沒有 `--write-back` 時只查資料與輸出 UI 格式；加上 `--write-back` 才會寫 DB。


In [ ]:
show_file("scripts/run_past_current_integrated_demo.py", start=1, end=130)


## 9. 和 0616ver_4 的差異

### 0616ver_4
```text
Future / Monitoring / Alert / DB 回寫主線完成。
past/current 尚未正式接進整合 service。
```

### 0616ver_5
```text
補 IntegratedSprayLineService：
1. past / current 資料取得
2. time slider 概念
3. current snapshot
4. past window
5. UI time-series 格式
6. future / alert / DB 回寫仍走 Database/versionB
```

### 報告講法
這版不是推翻 `0616ver_4`，而是在它上面補 past/current 前段資料流。


## 10. 最後總結稿

老師，我這版把 past/current/future 先整成同一個 service 入口。  
past/current 的資料取得和 time slider 概念有接進來，並輸出 UI 需要的 time-series 格式。

但正式 DB persistence 仍維持原本少榆端與余宇承 `Database/versionB` 的流程：

```text
alert_event -> db_alert
batch_station_status -> db_status
future_prediction_result -> db_future
```

所以這版同時做到：
1. 對齊 past/current 的 time-series 資料概念。
2. 保留少榆端 Future / Alert / Troubleshooting。
3. 不破壞 Database/versionB 的正式回寫流程。
